In [1]:
### !curl -L -o output_clean_date_technical.json "https://file.notion.so/f/f/d70b900c-92f2-4d32-870b-1fa0d80e953b/2cc1982f-a835-4d84-9002-318758475632/output_clean_date_technical.json?table=block&id=f447ef6f-695d-45bb-9e49-f6a9c2e5ddd0&spaceId=d70b900c-92f2-4d32-870b-1fa0d80e953b&expirationTimestamp=1758996000000&signature=mO2nf0bA_HLZtrPjOf22jztHsJiaGLvWOTioMjhaRWE&downloadName=output_clean_date_technical.json"
import pandas as pd
import json

# 讀取 JSON
with open("output_clean_date_technical.json", "r", encoding="utf-8") as f:
    data = json.load(f)

df_hist = pd.DataFrame(data['historicalPriceFull'])
historical_expanded = df_hist['historical'].apply(pd.Series)
p_df = pd.concat([df_hist['symbol'], historical_expanded], axis=1)
p_df['date'] = pd.to_datetime(p_df['date'])

def date_to_year_period(date):
    year = date.year
    month = date.month
    if 1 <= month <= 3:
        period = 'Q1'
    elif 4 <= month <= 6:
        period = 'Q2'
    elif 7 <= month <= 9:
        period = 'Q3'
    else:
        period = 'Q4'
    return pd.Series({'calendarYear': str(year), 'period': period})

p_df[['calendarYear', 'period']] = p_df['date'].apply(date_to_year_period)

keys = ['financialGrowth', 'ratios', 'cashFlowStatementGrowth', 'incomeStatementGrowth', 'balanceSheetStatementGrowth']

merged_df = p_df.copy()

for key in keys:
    df_fin = pd.DataFrame(data[key])
    df_fin['calendarYear'] = df_fin['calendarYear'].astype(str)
    df_fin['period'] = df_fin['period'].astype(str)
    merge_keys = ['symbol', 'calendarYear', 'period']
    df_fin.drop('date',axis=1,inplace=True)
    
    duplicate_cols = [col for col in df_fin.columns if col in merged_df.columns and col not in merge_keys]
    df_fin.rename(columns={col: f"{key}_{col}" for col in duplicate_cols}, inplace=True)
    
    merged_df = pd.merge(
    merged_df,
    df_fin,
    on=merge_keys,
    how='left',
    )

In [2]:
merged_df.isnull().sum()

symbol                                          0
date                                            0
open                                            0
high                                            0
low                                             0
                                               ..
growthTotalStockholdersEquity                  72
growthTotalLiabilitiesAndStockholdersEquity    72
growthTotalInvestments                         72
growthTotalDebt                                72
growthNetDebt                                  72
Length: 199, dtype: int64

In [3]:
merged_df['date'] = pd.to_datetime(merged_df['date'])

In [4]:
null_counts = merged_df.isnull().sum()
null_counts = null_counts[null_counts > 0].sort_values(ascending=False)
null_counts

revenueGrowth                72
growthOperatingCashFlow      72
growthFreeCashFlow           72
growthRevenue                72
growthCostOfRevenue          72
                             ..
fixedAssetTurnover           72
assetTurnover                72
operatingCashFlowPerShare    72
freeCashFlowPerShare         72
growthNetDebt                72
Length: 183, dtype: int64

In [5]:
def is_after_2023_Q3(row):
    year = int(row['calendarYear'])
    period = row['period']

    if year > 2023:
        return True

    if year == 2023:
        if period =='Q4':  
            return True

    return False

# 篩選排除 2023 Q3 之後的資料
filtered_df = merged_df[~merged_df.apply(is_after_2023_Q3, axis=1)]

In [6]:
sum(filtered_df.isnull().sum())

0

In [7]:
filtered_df.head()

,symbol,date,open,high,low,close,adjClose,volume,unadjustedVolume,change,...,growthTotalLiabilities,growthCommonStock,growthRetainedEarnings,growthAccumulatedOtherComprehensiveIncomeLoss,growthOthertotalStockholdersEquity,growthTotalStockholdersEquity,growthTotalLiabilitiesAndStockholdersEquity,growthTotalInvestments,growthTotalDebt,growthNetDebt
72,1101.TW,2023-09-28,33.1,33.30,33.05,33.25,33.25,18217658,18217658,0.15,...,0.02964,0.0,0.018448,0.0,0.093871,0.037866,0.036323,0.042211,0.053001,-0.110068
73,1101.TW,2023-09-27,33.0,33.20,32.90,33.05,33.05,15043077,15043077,0.05,...,0.02964,0.0,0.018448,0.0,0.093871,0.037866,0.036323,0.042211,0.053001,-0.110068
74,1101.TW,2023-09-26,33.1,33.30,33.00,33.00,33.00,18175159,18175159,-0.10,...,0.02964,0.0,0.018448,0.0,0.093871,0.037866,0.036323,0.042211,0.053001,-0.110068
75,1101.TW,2023-09-25,33.4,33.45,33.00,33.10,33.10,29278663,29278663,-0.30,...,0.02964,0.0,0.018448,0.0,0.093871,0.037866,0.036323,0.042211,0.053001,-0.110068
76,1101.TW,2023-09-22,33.5,33.60,33.35,33.50,33.50,23115549,23115549,0.00,...,0.02964,0.0,0.018448,0.0,0.093871,0.037866,0.036323,0.042211,0.053001,-0.110068


In [8]:
filtered_df.to_csv('clean_data.csv',index=False)

In [9]:
### filtered_df.columns

In [10]:
p_df

,symbol,date,open,high,low,close,adjClose,volume,unadjustedVolume,change,changePercent,vwap,label,changeOverTime,calendarYear,period
0,1101.TW,2024-01-12,33.70,33.80,33.60,33.75,33.750000,5221622,5221622,0.05,0.14837,33.72,"January 12, 24",0.001484,2024,Q1
1,1101.TW,2024-01-11,33.70,33.80,33.60,33.70,33.700000,6590499,6590499,0.00,0.00000,33.70,"January 11, 24",0.000000,2024,Q1
2,1101.TW,2024-01-10,34.05,34.05,33.70,33.70,33.700000,10231832,10231832,-0.35,-1.03000,33.82,"January 10, 24",-0.010300,2024,Q1
3,1101.TW,2024-01-09,34.30,34.30,34.05,34.05,34.050000,6191243,6191243,-0.25,-0.72886,34.13,"January 09, 24",-0.007289,2024,Q1
4,1101.TW,2024-01-08,34.40,34.55,34.25,34.25,34.250000,5522713,5522713,-0.15,-0.43605,34.35,"January 08, 24",-0.004360,2024,Q1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
731,1101.TW,2021-01-20,37.87,37.91,37.09,37.23,33.397839,38529978,38529978,-0.64,-1.69000,37.41,"January 20, 21",-0.016900,2021,Q1
732,1101.TW,2021-01-19,37.82,38.18,37.82,37.96,34.050301,13261660,13261660,0.14,0.37017,37.99,"January 19, 21",0.003702,2021,Q1
733,1101.TW,2021-01-18,38.14,38.14,37.64,37.82,33.927963,27598255,27598255,-0.32,-0.83901,37.87,"January 18, 21",-0.008390,2021,Q1
734,1101.TW,2021-01-15,38.64,38.64,38.14,38.14,34.213417,33681520,33681520,-0.50,-1.29000,38.31,"January 15, 21",-0.012900,2021,Q1
